# FireSystem: Hotspot Forecast from Leaf Area with XGBoost

Forecasts, for every 500 m pixel of a region and month, whether NASA FIRMS records an active fire **hotspot** (MODIS or VIIRS S-NPP), using **leaf area index** from two to three years earlier and the pixel's **land cover class** from two years earlier, and its **elevation and slope** as the features.

1. **Download hotspots**: FIRMS yearly country files (no key) and, for the months after the last yearly file, the FIRMS area API (`FIRMS_MAP_KEY` in `.env`). Only vegetation fires (`type` 0) with nominal / high confidence are kept; API rows have no `type`, so they are dropped in pixels where the yearly files show volcanoes, industry or offshore sources.
2. **Grid them** on the 500 m leaf area grid (MCD15A3H, `data/AppEEARS/<Region>/extra/`): label = 1 if the pixel has at least one hotspot in the month.
3. **Sample a pixel-month panel**: every pixel-month with a hotspot, plus `NEGATIVES_PER_POSITIVE` pixel-months without one per hotspot, drawn at the same rate in every month (so the no-hotspot rows keep the real time distribution and the predicted probabilities can be corrected back to real ones).
4. **Leaf area features** (each from months *t* − 24 … *t* − 35):

| Feature | Meaning |
|---|---|
| `lai_lag_24` … `lai_lag_35` | the pixel's monthly mean leaf area index (m²/m²) in month *t* − L |
| `lai_mean_24_35`, `lai_min_…`, `lai_max_…`, `lai_std_…` | statistics of those 12 months |
| `lai_valid_months_24_35` | months with a good-quality leaf area value |
| `elevation_m`, `slope_deg` | SRTM (90 m) elevation and slope averaged over the 500 m pixel; fixed in time, so they cannot leak |
| `land_cover` | MCD12Q1 `LC_Type1` IGBP class (1–17, categorical) of the latest year ≤ the row's year − `LAND_COVER_LAG_YEARS` (2); a fire can change its own year's class, so that year would leak |

   Land cover is read from the region's AppEEARS NetCDF (`MCD12Q1.061*.nc` in `data/AppEEARS/<Region>/`, from `AppEEARS_API_Area.ipynb`); it is skipped until that file exists. Optional (off by default): calendar month (`USE_CALENDAR`) and hotspot lags 12–24 months (`USE_HOTSPOT_LAGS`).
5. **Split chronologically** by fire season (July–June) into train → validation → test, **tune with walk-forward validation** on the validation seasons, evaluate walk-forward on the test seasons against two baselines (hotspot in the same month last year; the month's hotspot share).
6. **Forecast** the months after the last full month and save the model, scores and forecast (with the real probability).

## 1. Set Up

In [ ]:
import json
import sys
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils import chile_regions as cr
from src.utils import fire_datasets as fd
from src.utils import forecasting as fc
from src.utils import geotiff_layers as gl
from src.utils import hotspots as hs

try:
    import xgboost as xgb
except Exception as error:  # macOS needs the OpenMP runtime
    raise ImportError("XGBoost could not be loaded. On macOS install the OpenMP runtime: brew install libomp") from error
print("xgboost", xgb.__version__)

## 2. Set Parameters
`REGION` needs leaf area GeoTIFFs in `data/AppEEARS/<Region>/extra/` (reduced regions such as `Maule4` use their parent's folder and are cut to their own outline).

In [ ]:
# Region to forecast (e.g. "Maule", or "Maule4" for a quick run on part of Maule)
REGION = "Maule"

In [ ]:
SLUG = cr.region_slug(REGION)
PARENT_REGION = cr.REDUCED_REGIONS.get(REGION, (REGION, 1.0))[0]
DATA_DIR = REPO_ROOT / "data"
LAI_DIR = DATA_DIR / "AppEEARS" / cr.region_slug(PARENT_REGION) / "extra"
LAND_COVER_DIR = DATA_DIR / "AppEEARS" / cr.region_slug(PARENT_REGION)   # MCD12Q1 NetCDF of the region's AppEEARS task
FIRMS_ARCHIVE_DIR = DATA_DIR / "FIRMS" / "archive"          # yearly Chile files, shared by every region
FIRMS_API_DIR = DATA_DIR / "FIRMS" / "api" / SLUG             # API requests for this region's bounding box
OUTPUT_DIR = DATA_DIR / "models" / "hotspots" / SLUG    # forecast CSV, reports, dated model copy
MODEL_DIR = REPO_ROOT / "models"                            # model.json + metadata.json loaded by the API

START = date(2012, 3, 1)                                      # first month of hotspots and leaf area
LAST_MONTH = (date.today().replace(day=1) - timedelta(days=1)).replace(day=1)   # last full month
DOWNLOAD = True                                               # False: use the files already downloaded

LAI_LAGS = range(24, 36)         # leaf area from 2 to 3 years before the month
USE_LAND_COVER = True            # add the IGBP land cover class (if the MCD12Q1 file is in LAND_COVER_DIR)
LAND_COVER_LAG_YEARS = 2         # class of the latest year <= row year - 2
USE_TERRAIN = True               # add elevation / slope (SRTM DEM GeoTIFF in LAI_DIR)
USE_CALENDAR = False             # add month / quarter features
USE_HOTSPOT_LAGS = False         # add hotspot_lag_12 ... hotspot_lag_24
HOTSPOT_LAGS = range(12, 25)
NEGATIVES_PER_POSITIVE = 11      # no-hotspot pixel-months sampled per hotspot pixel-month
SEED = 42

SEASON_START_MONTH = 7           # fire seasons July-June
VAL_SEASONS = 2
TEST_SEASONS = 2
METRIC = "average_precision"     # "average_precision" (PR AUC), "roc_auc", "log_loss" or "brier"
PARAM_GRID = {
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
    "n_estimators": [300],
    "min_child_weight": [1, 10],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
    "reg_lambda": [1.0],
}

# Months ahead every feature is known: the shortest lag of each feature group. Land cover of year Y - L uses data
# up to December of that year, so for a January row it is only 12 * (L - 1) + 1 months old.
lags_months = [min(LAI_LAGS)]
if USE_HOTSPOT_LAGS:
    lags_months.append(min(HOTSPOT_LAGS))
if USE_LAND_COVER:
    lags_months.append(12 * (LAND_COVER_LAG_YEARS - 1) + 1)
HORIZON = min(lags_months)
FORECAST_MONTHS = 12             # months to forecast after LAST_MONTH (at most HORIZON)
FORECAST_STRIDE = 2              # forecast every n-th pixel in each direction
if FORECAST_MONTHS > HORIZON:
    raise ValueError(f"FORECAST_MONTHS ({FORECAST_MONTHS}) cannot exceed the features' horizon ({HORIZON} months)")


def make_model(params: dict) -> xgb.XGBClassifier:
    return xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist", enable_categorical=True,
        random_state=SEED, n_jobs=-1, **params,
    )


chile = cr.load_regions(DATA_DIR / "boundaries" / cr.BOUNDARIES_FILE)
region = cr.named_region_geometry(chile, REGION)
bbox = cr.geometry_bbox(region)
print(f"Region {REGION} | bbox {bbox} | {START} .. {LAST_MONTH} | leaf area from {LAI_DIR}")
print(f"Features known {HORIZON} months ahead; forecasting {FORECAST_MONTHS} months")

## 3. Download the Hotspots
Yearly files first (skipped when present); then the FIRMS area API for the days after the last yearly file, standard processing (SP) where available and near-real-time (NRT) after it. The API needs `FIRMS_MAP_KEY` and `FIRMS_BASE_URL` in `.env`; without them only the yearly files are used.

In [ ]:
archive = hs.download_hotspots(FIRMS_ARCHIVE_DIR, START.year, LAST_MONTH.year) if DOWNLOAD else sorted(FIRMS_ARCHIVE_DIR.glob("*_Chile.csv"))
archive_end = date(max(int(p.name.split("_")[1]) for p in archive), 12, 31)
print(f"{len(archive)} yearly files, up to {archive_end}")

api_files = sorted(FIRMS_API_DIR.glob("api_*.csv"))
if DOWNLOAD and archive_end < date.today():
    try:
        from src.utils import firms
        availability = {
            r["data_id"]: (date.fromisoformat(r["min_date"]), date.fromisoformat(r["max_date"]))
            for r in firms.fetch_sources() if r.get("data_id") in ("MODIS_SP", "MODIS_NRT", "VIIRS_SNPP_SP", "VIIRS_SNPP_NRT")
        }
        api_files = hs.download_api_hotspots(FIRMS_API_DIR, bbox, archive_end + timedelta(days=1), date.today(), availability)
        print(f"{len(api_files)} API requests after {archive_end}:", {k: f"{a}..{b}" for k, (a, b) in availability.items()})
    except KeyError as error:
        print(f"No FIRMS API key ({error}): using the yearly files only, up to {archive_end}")
paths = [*archive, *api_files]

## 4. Grid the Hotspots and Leaf Area

In [ ]:
spots = hs.load_hotspots(paths, region=region)
lai = gl.open_lai(LAI_DIR)
lat, lon = lai["lat"].values, lai["lon"].values
months = pd.date_range(START, LAST_MONTH, freq="MS")

inside = fd.region_mask(lai, region)
monthly_lai = gl.monthly_mean(lai, "lai", months)
valid = inside & np.isfinite(monthly_lai.values).any(axis=0)   # vegetated pixels of the region
static = hs.static_source_mask(paths, lat, lon)
grid = hs.monthly_hotspot_grid(spots, lat, lon, months, api_exclude=static)

terrain = None
if USE_TERRAIN:
    try:
        terrain = gl.terrain_on_grid(gl.open_elevation(LAI_DIR), lat, lon)
        print(f"Terrain: elevation {np.nanmin(terrain['elevation_m'].values):.0f}..{np.nanmax(terrain['elevation_m'].values):.0f} m on the 500 m grid")
    except FileNotFoundError:
        print(f"No SRTM DEM GeoTIFF in {LAI_DIR}: elevation / slope skipped")

land_cover = None
if USE_LAND_COVER:
    try:
        lc_ds = fd.open_product(fd.DATASETS["MODIS_land_cover"]["product"], LAND_COVER_DIR)
        land_cover = fd.land_cover_grid(lc_ds)["land_cover_igbp"].load()
        lc_years = pd.DatetimeIndex(land_cover["time"].values).year
        print(f"Land cover: {len(lc_years)} years ({lc_years.min()}..{lc_years.max()}) from {LAND_COVER_DIR}")
    except FileNotFoundError:
        print(f"No MCD12Q1 file in {LAND_COVER_DIR} yet: land cover skipped (run AppEEARS_API_Area.ipynb for {PARENT_REGION} first)")

print(f"{len(spots):,} hotspots ({spots['date'].min():%Y-%m-%d} .. {spots['date'].max():%Y-%m-%d}): {spots.groupby('sensor').size().to_dict()}")
print(f"{inside.sum():,} pixels in the region, {valid.sum():,} with leaf area; {static.sum():,} cells with non-vegetation sources")
per_year = pd.DataFrame({
    "hotspots": spots.groupby(spots["date"].dt.year).size(),
    "hotspot_pixel_months": pd.Series(((grid.values > 0) & valid).sum(axis=(1, 2)), index=months).groupby(months.year).sum(),
})
per_year

## 5. Sample the Pixel-Month Panel
Rows start once the full leaf area window is available (`START` + `max(LAI_LAGS)` months).

In [ ]:
first_month = pd.Timestamp(START) + pd.DateOffset(months=max(LAI_LAGS))
panel, NEGATIVE_RATE = hs.sample_panel(grid, valid, first_month, NEGATIVES_PER_POSITIVE, SEED)
full_panel_rows = int((months >= first_month).sum() * valid.sum())
print(f"Panel: {len(panel):,} rows ({panel['hotspot'].sum():,} with a hotspot) from {first_month:%Y-%m} to {months[-1]:%Y-%m}")
print(f"Full panel would be {full_panel_rows:,} pixel-months; negatives kept at rate {NEGATIVE_RATE:.4%}")

## 6. Build the Features

In [ ]:
def build_features(rows: pd.DataFrame) -> pd.DataFrame:
    out = pd.concat([
        rows,
        hs.lai_features(rows, monthly_lai, LAI_LAGS),
        hs.hotspot_lag_features(rows, grid, sorted({12, *HOTSPOT_LAGS})),
    ], axis=1)
    out["cal_month"] = pd.DatetimeIndex(out["date"]).month
    if terrain is not None:
        i, j = out["lat_idx"].to_numpy(dtype=int), out["lon_idx"].to_numpy(dtype=int)
        out["elevation_m"] = terrain["elevation_m"].values[i, j]
        out["slope_deg"] = terrain["slope_deg"].values[i, j]
    if land_cover is not None:
        out = pd.concat([out, hs.land_cover_features(out, land_cover, LAND_COVER_LAG_YEARS)], axis=1)
    if USE_CALENDAR:
        out = pd.concat([out, fc.add_calendar_features(out).drop(columns="month")], axis=1)
    return out


features = build_features(panel)
FEATURES = [c for c in features.columns if c.startswith("lai_")]
if terrain is not None:
    FEATURES += ["elevation_m", "slope_deg"]
if land_cover is not None:
    FEATURES += ["land_cover"]
if USE_CALENDAR:
    FEATURES += ["cal_month", "quarter", "month_sin", "month_cos"]
if USE_HOTSPOT_LAGS:
    FEATURES += [f"hotspot_lag_{lag}" for lag in HOTSPOT_LAGS]
print(f"{len(FEATURES)} features: {FEATURES}")
print("\nMissing share:")
print(features[FEATURES].isna().mean().round(3).to_string())
features.head()

In [ ]:
# Hotspot share per land cover class (sampled rows)
if land_cover is not None:
    by_class = features.groupby("land_cover", observed=True)["hotspot"].agg(rows="size", hotspot_share="mean")
    by_class.insert(0, "class", by_class.index.map(hs.IGBP_CLASSES))
    display(by_class.round(4))
else:
    print("Land cover not loaded")

## 7. Split Chronologically

In [ ]:
df, split_seasons = fc.chronological_split(features, VAL_SEASONS, TEST_SEASONS, SEASON_START_MONTH)
summary = df.groupby("split", sort=False).agg(
    first_date=("date", "min"), last_date=("date", "max"), rows=("hotspot", "size"), hotspot_share=("hotspot", "mean"),
).loc[["train", "val", "test"]]
summary["seasons"] = [split_seasons[s] for s in summary.index]
summary

## 8. Tune with Walk-Forward Validation
For each validation season: fit on all earlier seasons, predict that season. Test seasons are not used.

In [ ]:
train_val = df[df["split"] != "test"]
tuning = fc.tune_walk_forward(train_val, FEATURES, "hotspot", make_model, PARAM_GRID, split_seasons["val"], METRIC)
best_params = {k: tuning.iloc[0][k] for k in PARAM_GRID}
best_params = {k: int(v) if k in ("max_depth", "n_estimators") else float(v) for k, v in best_params.items()}
print("Best parameters:", best_params)
tuning

## 9. Evaluate on the Test Seasons (Walk-Forward)
Each test season is predicted by a model fitted on every earlier season. Baselines: **seasonal naive** (hotspot in the pixel in the same month last year) and **climatology** (hotspot share of the calendar month in the training rows). Scores are on the sampled rows (1 hotspot : `NEGATIVES_PER_POSITIVE` no-hotspot), so precision-type scores depend on that ratio; ROC AUC does not.

In [ ]:
runs = {
    "xgboost": (lambda: make_model(best_params), FEATURES),
    "seasonal_naive": (lambda: fc.SeasonalNaive("hotspot_lag_12"), ["hotspot_lag_12"]),
    "climatology": (lambda: fc.Climatology(by=("cal_month",)), ["cal_month"]),
}
test_predictions = {name: fc.walk_forward(df, cols, "hotspot", factory, split_seasons["test"]) for name, (factory, cols) in runs.items()}
comparison = pd.DataFrame({name: fc.score_predictions(p).loc["all"] for name, p in test_predictions.items()}).T.astype(float).round(4)
comparison

In [ ]:
fc.score_predictions(test_predictions["xgboost"]).astype(float).round(4)

## 10. Fit the Final Model

In [ ]:
final_model = make_model(best_params).fit(df[FEATURES], df["hotspot"])
importance = pd.Series(final_model.get_booster().get_score(importance_type="gain")).reindex(FEATURES).fillna(0)
(importance / importance.sum()).sort_values(ascending=False).round(4).rename("gain_share").to_frame()

## 11. Forecast
Every `FORECAST_STRIDE`-th vegetated pixel for the `FORECAST_MONTHS` months after `LAST_MONTH`. `hotspot_score` is the model output on the sampled scale; `hotspot_probability` undoes the negative sampling (real odds = sampled odds × `NEGATIVE_RATE`) and estimates the real chance of a hotspot in the pixel and month.

In [ ]:
future = build_features(hs.future_panel(valid, months, lat, lon, FORECAST_MONTHS, FORECAST_STRIDE))
future["hotspot_score"] = final_model.predict_proba(future[FEATURES])[:, 1]
future["hotspot_probability"] = hs.corrected_probability(future["hotspot_score"], NEGATIVE_RATE)
future.insert(0, "region", REGION)

monthly_forecast = future.groupby(future["date"].dt.to_period("M")).agg(
    pixels=("hotspot_probability", "size"),
    mean_probability=("hotspot_probability", "mean"),
    p99_probability=("hotspot_probability", lambda s: s.quantile(0.99)),
    expected_hotspot_pixels=("hotspot_probability", "sum"),
)
monthly_forecast["expected_hotspot_pixels_full_grid"] = monthly_forecast["expected_hotspot_pixels"] * FORECAST_STRIDE ** 2
monthly_forecast.round(5)

## 12. Save the Model, Scores and Forecast
Also saves `model.json` and `metadata.json` (features, lags, grid, negative sampling rate) in `MODEL_DIR` (`models/` at the repository root): the files the API endpoint `/api/v1/hotspot/forecast` loads. The forecast CSV, reports and dated model copy stay in `OUTPUT_DIR` (`data/`), which is not in git.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
period = f"{months[0]:%Y%m}_{months[-1]:%Y%m}"
model_path = OUTPUT_DIR / f"xgboost_hotspots_{SLUG}_{period}.json"
final_model.save_model(model_path)

forecast = future[["region", "date", "lat", "lon", "hotspot_score", "hotspot_probability", *FEATURES]].copy()
forecast.attrs["description"] = {
    "title": f"Monthly hotspot forecast ({REGION})",
    "label": "1 = at least one FIRMS MODIS / VIIRS S-NPP vegetation-fire hotspot (nominal / high confidence) in the 500 m pixel and month",
    "hotspot_score": "XGBoost output on the sampled scale (1 hotspot : NEGATIVES_PER_POSITIVE no-hotspot rows)",
    "hotspot_probability": "score corrected for negative sampling: estimated real probability for the pixel and month",
    "negative_rate": NEGATIVE_RATE,
    "model": model_path.name,
    "features": FEATURES,
    "horizon_months": HORIZON,
    "spatial_granularity": f"500 m pixels, every {FORECAST_STRIDE}-th in each direction",
    "update_frequency": "Monthly",
    "source": "NASA FIRMS (hotspots), AppEEARS MCD15A3H (leaf area index), SRTMGL3 (elevation), MCD12Q1 (land cover)",
}
saved = fd.save_dataframe(forecast, OUTPUT_DIR / f"forecast_hotspots_{SLUG}_{period}", "csv")

report = {
    "region": REGION, "months": [str(months[0].date()), str(months[-1].date())], "features": FEATURES,
    "panel_rows": len(panel), "hotspot_rows": int(panel["hotspot"].sum()), "negative_rate": NEGATIVE_RATE,
    "best_params": best_params, "split_seasons": {k: [int(s) for s in v] for k, v in split_seasons.items()},
    "test_comparison": json.loads(comparison.to_json(orient="index")),
}
report_path = OUTPUT_DIR / f"report_hotspots_{SLUG}_{period}.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
tuning.to_csv(OUTPUT_DIR / f"tuning_hotspots_{SLUG}_{period}.csv", index=False)

# Latest model for the API (/api/v1/hotspot/forecast): model.json + metadata.json with what is needed
# to rebuild the features of any pixel and month exactly as in training
MODEL_DIR.mkdir(parents=True, exist_ok=True)
final_model.save_model(MODEL_DIR / "model.json")
metadata = {
    "region": REGION,
    "bbox": list(bbox),
    "trained_months": [f"{first_month:%Y-%m}", f"{months[-1]:%Y-%m}"],
    "features": FEATURES,
    "lai_lags": list(LAI_LAGS),
    "use_terrain": terrain is not None,
    "use_land_cover": land_cover is not None,
    "land_cover_lag_years": LAND_COVER_LAG_YEARS,
    "land_cover_categories": list(hs.IGBP_CLASSES),
    "negative_rate": NEGATIVE_RATE,
    "horizon_months": HORIZON,
    "grid": {"lat": lat.tolist(), "lon": lon.tolist()},
    "params": best_params,
    "test_comparison": json.loads(comparison.to_json(orient="index")),
    "xgboost_version": xgb.__version__,
}
(MODEL_DIR / "metadata.json").write_text(json.dumps(metadata, indent=1, ensure_ascii=False), encoding="utf-8")
print("Saved:", model_path, *saved, report_path, MODEL_DIR / "model.json", MODEL_DIR / "metadata.json", sep="\n  ")